# Unity 카탈로그에 데이터 수집

## 산업 시나리오: GreenGrid Energy
**GreenGrid Energy**는 유럽과 북미 전역에서 풍력 발전소와 태양광 발전소를 운영하는 재생 에너지 회사입니다. 데이터 엔지니어링 팀은 여러 소스의 데이터를 Unity 카탈로그로 수집해야 합니다.
- **터빈 센서 판독값** — 지속적인 전력 출력과 날씨 지표를 생성하는 IoT 장치
- **운영 이벤트 피드** — 터빈 상태 전환을 추적하는 SCADA 시스템의 변경 데이터
- **시장 가격 파일** — 클라우드 스토리지에 CSV 배치로 전달되는 시간별 에너지 현물 가격
- **유지보수 기록** — 각 서비스 방문 후 현장 직원이 JSON 파일로 업로드합니다.
이 노트북은 GreenGrid 데이터를 실행 예제로 사용하여 모듈에서 다루는 각 수집 기술을 안내합니다.

### 설정
설정 스크립트는 데모에서 참조하는 다음 테이블을 사용하여 `trainer_demo.demo_07` 스키마를 생성합니다.
 Table | 설명 |
---|---|
 `energy_sites` | 풍력 및 태양광 발전소 위치 (20개 사이트) |
 `turbines` | 개별 터빈/패널 자산 (200개 레코드) |
 `turbine_readings` | 시간별 발전량 판독값 (50,000행) |
 `turbine_events` | CDC 스타일 변경 피드 — 터빈 상태 전환 |
 `market_prices` | 지역별 시간별 에너지 현물 가격 |

모든 테이블을 초기화하려면 아래 셀을 실행하세요.

In [0]:
from scripts.setup import setup, setup_07

spark.sql("USE CATALOG trainer_demo")
setup_07(spark)

## Lakeflow Connect로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-lakeflow-connect.png)

### 🎯 데모: Lakeflow Connect 수집 파이프라인 구성
> **강사 참고 사항:** Lakeflow Connect는 Azure Databricks UI 또는 자산 번들을 통해 구성됩니다. 실행 가능한 노트북 코드는 없습니다. UI 마법사를 살펴보거나 아래 YAML 정의를 표시하세요.
다음 에셋 번들 구성은 온프레미스 SQL Server SCADA 시스템의 `turbine_readings` 테이블을 Unity 카탈로그로 수집합니다. **SCD 유형 2**를 사용하여 터빈 상태 변경의 전체 기록을 보존하고 **CDC**를 사용하여 실행 간에 수정된 행만 추적합니다.
```yaml
resources:
  pipelines:
    gateway_greengrid:
      name: greengrid-scada-gateway
      gateway_definition:
        connection_name: scada-sql-server-prod
        gateway_storage_catalog: trainer_demo
        gateway_storage_schema: demo_07
        gateway_storage_name: greengrid_gateway

    pipeline_turbine_readings:
      name: greengrid-turbine-readings-pipeline
      catalog: trainer_demo
      schema: demo_07
      ingestion_definition:
        ingestion_gateway_id: ${resources.pipelines.gateway_greengrid.id}
        objects:
          - table:
              source_schema: dbo
              source_table: turbine_readings
              destination_catalog: trainer_demo
              destination_schema: demo_07
              table_configuration:
                scd_type: SCD_TYPE_2
                sequence_by: reading_ts
                include_columns:
                  - turbine_id
                  - reading_ts
                  - power_output_kw
                  - wind_speed_ms
                  - operational_status
```

강조할 핵심 사항:
- **수집 게이트웨이**는 SQL Server 원본에서 변경 데이터를 지속적으로 읽습니다.
- `scd_type: SCD_TYPE_2`은 행 기록을 추적하기 위해 `__START_AT` 및 `__END_AT` 열을 추가합니다.
- `include_columns`은 데이터 전송을 제한합니다. 새 소스 열은 자동으로 표시되지 않습니다.

## 노트북으로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-notebooks.png)

### 🎯 데모: DataFrames를 사용하여 파일에서 터빈 판독값 수집
GreenGrid의 현장 팀은 정기적으로 터빈 판독값을 CSV 파일로 Unity 카탈로그 볼륨으로 내보냅니다. 이 데모에서는 해당 파일을 DataFrame으로 읽고 Delta 테이블에 쓰는 방법을 보여줍니다.

In [0]:
# Write sample CSV data to a Unity Catalog Volume for the demo
# (Simulates files dropped by the field export process)

sample_csv = (
    "turbine_id,reading_ts,power_output_kw,wind_speed_ms,temperature_c,operational_status\n"
    "TRB-001,2024-03-01 06:00:00,1850.5,8.2,12.3,operational\n"
    "TRB-001,2024-03-01 07:00:00,2100.0,9.1,11.8,operational\n"
    "TRB-002,2024-03-01 06:00:00,0.0,2.1,10.5,offline\n"
    "TRB-002,2024-03-01 07:00:00,950.3,6.7,11.0,operational\n"
    "TRB-003,2024-03-01 06:00:00,3200.8,12.4,9.7,operational\n"
    "TRB-003,2024-03-01 07:00:00,3150.2,11.9,9.5,operational\n"
    "TRB-004,2024-03-01 06:00:00,1200.1,7.3,13.1,maintenance\n"
    "TRB-005,2024-03-01 06:00:00,2800.9,10.8,8.9,operational\n"
)

dbutils.fs.put(
    "/Volumes/trainer_demo/demo_07/sensor_data_landing/readings_2024_03_01.csv",
    sample_csv,
    overwrite=True
)
print("Sample CSV written to volume")

In [0]:
# Read the CSV file into a DataFrame
df_readings = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("/Volumes/trainer_demo/demo_07/sensor_data_landing/readings_2024_03_01.csv")
)

df_readings.printSchema()
display(df_readings)

파일을 읽은 후 Unity 카탈로그 델타 테이블에 씁니다.
`mode` 매개변수는 기존 데이터가 처리되는 방식을 제어합니다.
- `overwrite` — 모든 행을 교체합니다(전체 새로 고침에 적합).
- `append` — 기존 테이블에 행을 추가합니다.
- `ignore` — 테이블이 이미 존재하는 경우 건너뜁니다.

In [0]:
# Write to a Unity Catalog Delta table (append new batch)
df_readings.write.mode("append").saveAsTable("trainer_demo.demo_07.turbine_readings_batch")

spark.sql("SELECT * FROM trainer_demo.demo_07.turbine_readings_batch").display()

#### VARIANT 유형으로 반구조화된 데이터 읽기
GreenGrid는 또한 구조가 터빈 모델에 따라 달라지는 유지 관리 JSON 페이로드도 수신합니다. **VARIANT** 유형은 원시 JSON을 저장하고 고정 스키마를 미리 정의하지 않고도 개별 필드를 쿼리할 수 있게 해줍니다.

In [0]:
from pyspark.sql.functions import col, expr

# Simulate JSON maintenance records with varying fields per turbine model
maintenance_json = spark.createDataFrame([
    ('{"turbine_id":"TRB-001","component":"gearbox","action":"lubrication","technician":"J.Smith","duration_hrs":2.5}',),
    ('{"turbine_id":"TRB-002","component":"blade","action":"inspection","remark":"surface erosion noted"}',),
], ["raw_payload"])

# Parse to VARIANT and extract specific fields
df_variant = maintenance_json.select(
    expr("parse_json(raw_payload)").alias("data")
).select(
    expr("variant_get(data, '$.turbine_id', 'STRING')").alias("turbine_id"),
    expr("variant_get(data, '$.component', 'STRING')").alias("component"),
    expr("variant_get(data, '$.action', 'STRING')").alias("action"),
    expr("variant_get(data, '$.technician', 'STRING')").alias("technician"),  # nullable
)

display(df_variant)

## SQL 메소드로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-sql-methods.png)

### 🎯 데모: SQL 메서드를 사용하여 테이블 수집 및 새로 고침
SQL 명령은 수집에 대한 선언적이고 읽기 쉬운 접근 방식을 제공합니다. 이 데모에서는 GreenGrid 터빈 데이터를 사용하는 세 가지 보완 기술을 보여줍니다.

#### CTAS — 생산 판독 테이블에서 브론즈 레이어 생성

In [0]:
%sql
-- CTAS: combine table creation and data population in one statement
-- Useful for initial loads and one-time schema migrations

CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.bronze_turbine_readings
AS
SELECT
    turbine_id,
    reading_ts,
    ROUND(power_output_kw, 2)    AS power_output_kw,
    wind_speed_ms,
    UPPER(operational_status)   AS operational_status,
    CURRENT_TIMESTAMP()         AS ingested_at
FROM trainer_demo.demo_07.turbine_readings
WHERE operational_status != 'decommissioned';

SELECT COUNT(*) AS row_count FROM trainer_demo.demo_07.bronze_turbine_readings;


#### 테이블 생성 또는 교체 — 집계된 일일 지표 새로 고침
파생 테이블을 완전히 바꾸려는 경우 이 기능을 사용하세요. 삭제 및 재생성과 달리 `CREATE OR REPLACE`은 부여된 권한과 행 필터를 유지합니다.

In [0]:
%sql
-- CREATE OR REPLACE TABLE: full refresh of an aggregated metrics table
-- Replaces all existing rows atomically; permissions and policies are preserved

CREATE OR REPLACE TABLE trainer_demo.demo_07.daily_site_production
AS
SELECT
    t.site_id,
    s.site_name,
    s.energy_type,
    DATE(r.reading_ts)                  AS production_date,
    SUM(r.power_output_kw)              AS total_output_kw,
    AVG(r.power_output_kw)              AS avg_output_kw,
    COUNT(DISTINCT r.turbine_id)        AS active_turbines
FROM trainer_demo.demo_07.turbine_readings r
JOIN trainer_demo.demo_07.turbines       t  ON r.turbine_id = t.turbine_id
JOIN trainer_demo.demo_07.energy_sites   s  ON t.site_id    = s.site_id
WHERE r.operational_status = 'operational'
GROUP BY t.site_id, s.site_name, s.energy_type, DATE(r.reading_ts);

SELECT * FROM trainer_demo.demo_07.daily_site_production
ORDER BY production_date DESC
LIMIT 10;


#### COPY INTO — 볼륨에서 새 CSV 파일을 증분식으로 로드합니다.
`COPY INTO`은 멱등성입니다. 이미 로드된 파일은 자동으로 추적되고 건너뛰므로 소스 디렉터리에 파일이 누적되더라도 일정에 따라 안전하게 실행할 수 있습니다.

In [0]:
# Simulate market price CSV files arriving daily from an energy exchange.
# GreenGrid receives hourly spot prices for each regional grid operator as
# a separate CSV file per day — one file per day is common from market operators.

import random
from datetime import date

regions = ['DE', 'FR', 'GB', 'US-CAISO', 'US-ERCOT']
base_prices = {'DE': 65.0, 'FR': 60.0, 'GB': 70.0, 'US-CAISO': 45.0, 'US-ERCOT': 38.0}

price_id = 1
for batch_day in range(3):
    price_date = date(2024, 3, batch_day + 1)
    rows = ["price_id,region,price_date,hour_utc,spot_price_eur,currency"]
    for region in regions:
        for hour in range(24):
            # Higher prices during peak hours (08:00–20:00 UTC)
            peak_factor = 1.3 if 8 <= hour <= 20 else 0.8
            price = round(base_prices[region] * peak_factor * (1 + random.uniform(-0.1, 0.1)), 2)
            currency = 'USD' if region.startswith('US') else 'EUR'
            rows.append(f"{price_id},{region},{price_date},{hour},{price},{currency}")
            price_id += 1

    dbutils.fs.put(
        f"/Volumes/trainer_demo/demo_07/market_prices_landing/prices_{price_date}.csv",
        "\n".join(rows),
        overwrite=True
    )

print("Market price CSV files written to landing volume.")
display(dbutils.fs.ls('/Volumes/trainer_demo/demo_07/market_prices_landing/'))


In [0]:
%sql
-- Ensure the target table exists with the expected schema
CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.market_prices_bronze (
    price_id        BIGINT,
    region          STRING,
    price_date      DATE,
    hour_utc        INT,
    spot_price_eur  DOUBLE,
    currency        STRING,
    source_file     STRING
);

-- COPY INTO: load CSV files from the volume (idempotent — already-loaded files are skipped)
COPY INTO trainer_demo.demo_07.market_prices_bronze
FROM (
    SELECT
        CAST(price_id AS BIGINT)        AS price_id,
        region,
        CAST(price_date AS DATE)        AS price_date,
        CAST(hour_utc AS INT)           AS hour_utc,
        CAST(spot_price_eur AS DOUBLE)  AS spot_price_eur,
        currency,
        _metadata.file_name             AS source_file
    FROM '/Volumes/trainer_demo/demo_07/market_prices_landing/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false');

SELECT COUNT(*) AS rows_loaded FROM trainer_demo.demo_07.market_prices_bronze;


## CDC 피드로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-cdc-feed.png)

### 🎯 데모: AUTO CDC API를 사용하여 운영 상태 변경 사항 수집
GreenGrid의 SCADA 시스템은 구조화된 변경 기록으로 터빈 상태 변경을 지속적으로 제공합니다. 각 레코드에는 터빈 데이터, 작업 유형(`INSERT`, `UPDATE`, `DELETE`) 및 주문을 위한 시퀀스 번호가 포함되어 있습니다.
Lakeflow Spark 선언적 파이프라인의 **AUTO CDC API**는 중복 제거, 순서가 잘못된 이벤트 처리, SCD 로직을 자동으로 처리합니다.
> **강사 참고 사항:** `dp.create_auto_cdc_flow`은 Lakeflow 파이프라인 정의 내에서 실행됩니다. 아래 셀은 노트북에서 직접 실행하여 이전/이후 결과를 표시할 수 있는 MERGE 기반 접근 방식을 사용하는 CDC 패턴을 보여줍니다.

In [0]:
# Simulate a CDC change feed from the SCADA system
# Each row represents a change event captured from the source database

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

cdc_schema = StructType([
    StructField("operation",          StringType(),   False),  # INSERT / UPDATE / DELETE
    StructField("sequence_num",       IntegerType(),  False),  # Monotonically increasing
    StructField("turbine_id",         StringType(),   True),
    StructField("site_id",            StringType(),   True),
    StructField("capacity_kw",        DoubleType(),   True),
    StructField("operational_status", StringType(),   True),
    StructField("last_updated_ts",    StringType(),   True),
])

cdc_changes = [
    Row("INSERT", 1, "TRB-901", "SITE-01", 3500.0, "commissioning", "2024-03-01 08:00:00"),
    Row("INSERT", 2, "TRB-902", "SITE-01", 3500.0, "commissioning", "2024-03-01 08:01:00"),
    Row("UPDATE", 3, "TRB-901", "SITE-01", 3500.0, "operational",   "2024-03-01 10:00:00"),
    Row("UPDATE", 4, "TRB-902", "SITE-01", 3500.0, "offline",       "2024-03-02 06:00:00"),
    Row("UPDATE", 5, "TRB-901", "SITE-01", 3500.0, "maintenance",   "2024-03-05 09:00:00"),
    Row("DELETE", 6, "TRB-902", "SITE-01", None,    None,           "2024-03-10 12:00:00"),
]

cdc_df = spark.createDataFrame(cdc_changes, schema=cdc_schema)
display(cdc_df)

#### SCD 유형 1 - 현재 상태만 유지
SCD 유형 1은 업데이트가 도착하면 기존 레코드를 덮어씁니다. 각 터빈의 최신 버전만 저장되며 기록 행은 없습니다.

In [0]:
# Create the target table for SCD Type 1
spark.sql(
    "CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.turbines_current ("
    "    turbine_id          STRING NOT NULL,"
    "    site_id             STRING,"
    "    capacity_kw         DOUBLE,"
    "    operational_status  STRING,"
    "    last_updated_ts     STRING"
    ")"
)

upserts = cdc_df.filter("operation != 'DELETE'").select(
    "turbine_id","site_id","capacity_kw","operational_status","last_updated_ts"
)
deletes = cdc_df.filter("operation = 'DELETE'").select("turbine_id")

upserts.createOrReplaceTempView("cdc_upserts")
deletes.createOrReplaceTempView("cdc_deletes")

spark.sql("""
    MERGE INTO trainer_demo.demo_07.turbines_current AS target
    USING cdc_upserts AS source
    ON target.turbine_id = source.turbine_id
    WHEN MATCHED     THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
    MERGE INTO trainer_demo.demo_07.turbines_current AS target
    USING cdc_deletes AS source
    ON target.turbine_id = source.turbine_id
    WHEN MATCHED THEN DELETE
""")

spark.sql("SELECT * FROM trainer_demo.demo_07.turbines_current ORDER BY turbine_id").display()

#### SCD 유형 2 — 전체 변경 내역을 보존합니다.
SCD 유형 2는 `__START_AT` 및 `__END_AT` 열을 추가합니다. 터빈 상태가 변경되면 이전 행이 닫히고(종료 날짜 설정) 새 행이 삽입됩니다. 분석가는 언제든지 각 터빈의 정확한 상태를 재구성할 수 있습니다.

In [0]:
# Create SCD Type 2 target table with history columns
spark.sql("""
    CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.turbines_history (
        turbine_id          STRING NOT NULL,
        site_id             STRING,
        capacity_kw         DOUBLE,
        operational_status  STRING,
        last_updated_ts     STRING,
        __START_AT          STRING,
        __END_AT            STRING
    )
""")

# The AUTO CDC API (inside a Lakeflow Pipeline) implements this automatically:
#
#   dp.create_streaming_table('turbines_history')
#   dp.create_auto_cdc_flow(
#       target              = 'turbines_history',
#       source              = 'cdc_feed_view',
#       keys                = ['turbine_id'],
#       sequence_by         = col('sequence_num'),
#       apply_as_deletes    = col('operation') == 'DELETE',
#       except_column_list  = ['operation', 'sequence_num'],
#       stored_as_scd_type  = 2
#   )

# For demonstration, manually populate the SCD2 output:
history_rows = [
    ("TRB-901","SITE-01",3500.0,"commissioning","2024-03-01 08:00:00","2024-03-01 08:00:00","2024-03-01 10:00:00"),
    ("TRB-901","SITE-01",3500.0,"operational",  "2024-03-01 10:00:00","2024-03-01 10:00:00","2024-03-05 09:00:00"),
    ("TRB-901","SITE-01",3500.0,"maintenance",  "2024-03-05 09:00:00","2024-03-05 09:00:00",None),
    ("TRB-902","SITE-01",3500.0,"commissioning","2024-03-01 08:01:00","2024-03-01 08:01:00","2024-03-02 06:00:00"),
    ("TRB-902","SITE-01",3500.0,"offline",      "2024-03-02 06:00:00","2024-03-02 06:00:00","2024-03-10 12:00:00"),
]
history_df = spark.createDataFrame(
    history_rows,
    ["turbine_id","site_id","capacity_kw","operational_status","last_updated_ts","__START_AT","__END_AT"]
)
history_df.write.mode("append").saveAsTable("trainer_demo.demo_07.turbines_history")

spark.sql("""
    SELECT turbine_id, operational_status, __START_AT, __END_AT,
           CASE WHEN __END_AT IS NULL THEN 'current' ELSE 'historical' END AS row_type
    FROM trainer_demo.demo_07.turbines_history
    ORDER BY turbine_id, __START_AT
""").display()

## Spark 구조적 스트리밍으로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-spark-structured-streaming.png)

### 🎯 데모: Spark Structured Streaming으로 실시간 터빈 센서 데이터 수집
GreenGrid의 풍력 터빈은 몇 초마다 판독값을 내보냅니다. 구조적 스트리밍은 이러한 이벤트를 지속적으로 캡처하여 델타 테이블에 저장합니다.
> **강사 참고 사항:** 내장된 `rate` 소스를 사용하여 실시간으로 도착하는 센서 이벤트를 시뮬레이션합니다. 프로덕션 환경에서 GreenGrid는 Kafka 호환 인터페이스를 통해 **Azure Event Hubs**에 연결됩니다.

In [0]:
# Simulate real-time turbine sensor events using the built-in rate source
# The rate source generates (timestamp, value) rows at a configurable rate

from pyspark.sql.functions import rand, when, expr, round as spark_round

simulated_readings = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .load()
        .withColumn(
            "turbine_id",
            expr("concat('TRB-', lpad(cast((value % 10) + 1 as string), 3, '0'))")
        )
        .withColumn("power_output_kw",    spark_round(rand() * 3500, 2))
        .withColumn("wind_speed_ms",      spark_round(rand() * 15 + 2, 1))
        .withColumn("temperature_c",      spark_round(rand() * 20 + 5, 1))
        .withColumn("operational_status", when(rand() > 0.95, "warning").otherwise("operational"))
        .select("timestamp","turbine_id","power_output_kw","wind_speed_ms","temperature_c","operational_status")
)

simulated_readings.printSchema()

In [0]:
# Write the streaming data to a Delta table
# Each streaming job must have its own unique checkpoint location

checkpoint_path = "/Volumes/trainer_demo/demo_07/sensor_data_landing/checkpoints/turbine_stream"

query = (
    simulated_readings.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable("trainer_demo.demo_07.turbine_stream_bronze")
)

print(f'Streaming query started: {query.id}')
print('Stream is running — run the next cell to check progress, then stop.')

In [0]:
# Check stream status and recent data
print(query.status)

spark.sql("""
    SELECT COUNT(*) AS rows_ingested, MAX(timestamp) AS latest_ts
    FROM trainer_demo.demo_07.turbine_stream_bronze
""").display()

# Stop the stream once continuous ingestion has been demonstrated
query.stop()
print('Stream stopped.')

#### 프로덕션에서 Azure Event Hubs에 연결
프로덕션 GreenGrid 환경에서 터빈은 판독값을 **Azure Event Hubs**로 보냅니다. Kafka 호환 인터페이스는 SASL 인증을 사용합니다. **Databricks Secrets**에 연결 문자열을 저장합니다. 자격 증명을 하드코딩하지 마세요.
```python
eh_namespace      = "greengrid-telemetry"
eh_name           = "turbine-readings"
connection_string = dbutils.secrets.get(scope="eventhubs", key="connection-string")

kafka_options = {
    "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
    "subscribe": eh_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="$ConnectionString" password="{connection_string}";'
    ),
    "startingOffsets": "latest",
}

df_eh = spark.readStream.format('kafka').options(**kafka_options).load()
```

> **핵심 사항:** 각 스트리밍 쿼리에는 고유한 체크포인트 위치가 필요합니다. 쿼리가 다시 시작되면 마지막으로 커밋된 체크포인트에서 다시 시작됩니다. 즉, 데이터가 손실되거나 중복되지 않습니다.

## 자동 로더로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-auto-loader.png)

### 🎯 데모: 자동 로더를 사용한 증분 파일 수집
현장 직원은 각 서비스 방문을 완료할 때마다 유지 관리 보고서 CSV 파일을 Unity 카탈로그 볼륨에 업로드합니다. 자동 로더는 랜딩 볼륨을 감시하고 작업이 다시 시작되더라도 각 파일을 정확히 한 번만 처리합니다.

In [0]:
# Simulate three batches of maintenance report files arriving in the landing volume

import random
from datetime import datetime, timedelta

for batch_num in range(1, 4):
    rows = ["turbine_id,maintenance_date,service_type,technician,cost_eur,notes"]
    for i in range(5):
        tid  = f"TRB-{random.randint(1, 200):03d}"
        dt   = (datetime(2024, 3, batch_num) + timedelta(hours=i)).strftime('%Y-%m-%d %H:%M')
        svc  = random.choice(['blade_inspection','gearbox_oil','electrical_check','sensor_calibration'])
        tech = random.choice(['J.Smith','A.Patel','M.Koster','L.Nguyen'])
        cost = round(random.uniform(200, 2500), 2)
        rows.append(f'{tid},{dt},{svc},{tech},{cost},routine check')

    dbutils.fs.put(
        f"/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_{batch_num:02d}.csv",
        "\n".join(rows),
        overwrite=True
    )

print('3 maintenance batches written to the landing volume')
display(dbutils.fs.ls('/Volumes/trainer_demo/demo_07/sensor_data_landing/'))

In [0]:
# Configure Auto Loader to incrementally ingest CSV files from the landing volume
# cloudFiles.format   — specifies the source file format
# cloudFiles.schemaLocation — stores the inferred schema for evolution tracking

base_path       = "/Volumes/trainer_demo/demo_07/sensor_data_landing"
schema_path     = f"{base_path}/.autoloader_schema"
checkpoint_path = f"{base_path}/.autoloader_checkpoint"

autoloader_query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",          "csv")
        .option("cloudFiles.schemaLocation",   schema_path)
        .option("cloudFiles.inferColumnTypes", "true")  # detect date, numeric types
        .option("header",                      "true")
        .load(f"{base_path}/maintenance_batch_*.csv")  # pattern-match only maintenance files
    .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema",         "true")          # allow schema evolution
        .trigger(availableNow=True)                     # process all pending files then stop
        .toTable("trainer_demo.demo_07.maintenance_bronze")
)

autoloader_query.awaitTermination()
print('Files processed:')
spark.sql("SELECT COUNT(*) AS total_rows FROM trainer_demo.demo_07.maintenance_bronze").display()

#### 자동 로더를 사용한 스키마 진화
현장 직원이 내보내기에 새 열(예: `gps_coordinates` 필드)을 추가하면 자동 로더는 변경 사항을 감지하고 `schemaEvolutionMode` 옵션을 사용하여 대상 스키마를 자동으로 발전시킬 수 있습니다.
| Mode | Behaviour |
|---|---|
| `addNewColumns` *(default)* | Stream **fails** on new column, adds it to the schema, **restarts cleanly** |
| `rescue` | New columns land in `_rescued_data` — stream never fails |
| `failOnNewColumns` | Stream fails and does not restart until schema is manually updated |
| `none` | New columns are silently dropped |

```python
# Use rescue mode to prevent stream failures on ad-hoc column additions
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",             "csv")
    .option("cloudFiles.schemaLocation",     schema_path)
    .option("cloudFiles.schemaEvolutionMode","rescue")  # unknowns go to _rescued_data
    .option("header",                        "true")
    .load(base_path))
```

`_rescued_data` 열은 예상 스키마와 일치하지 않는 모든 데이터를 캡처하여 파이프라인을 안정적으로 유지하면서 데이터 손실을 방지합니다.

## Lakeflow Spark 선언적 파이프라인으로 데이터 수집

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-lakeflow-declarative-pipelines.png)

### 🎯 데모: Lakeflow Spark 선언적 파이프라인을 사용하여 수집 파이프라인 정의
Lakeflow Spark 선언적 파이프라인을 사용하면 수집 파이프라인의 구조를 코드로 선언할 수 있습니다. 프레임워크는 오케스트레이션, 재시도, 검사점 및 정확히 한 번 의미 체계를 자동으로 처리합니다.
> **강사 참고 사항:** 아래 코드는 **파이프라인 소스 코드**입니다. Databricks UI에서 새 파이프라인을 만들고, 이러한 정의가 포함된 노트북이나 폴더를 가리키고, 노트북에서 직접 실행하지 않고 파이프라인 인터페이스에서 실행하세요.

In [0]:
# === PIPELINE SOURCE CODE (Python) ===
# Attach this notebook to a Lakeflow Pipeline — do not run cells directly

from pyspark import pipelines as dp
from pyspark.sql.functions import col

# ── Bronze: Auto Loader — new maintenance CSV files from the landing volume ──────
@dp.table
def maintenance_bronze():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format",         "csv")
            .option("cloudFiles.inferColumnTypes","true")
            .option("header",                    "true")
            .load("/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_*.csv")
    )

# ── Silver: CDC — apply turbine status changes as SCD Type 2 ─────────────────────
@dp.view
def turbine_cdc_feed():
    return spark.readStream.table("trainer_demo.demo_07.turbine_events")

dp.create_streaming_table("turbines_silver")

dp.create_auto_cdc_flow(
    target              = "turbines_silver",
    source              = "turbine_cdc_feed",
    keys                = ["turbine_id"],
    sequence_by         = col("sequence_num"),
    apply_as_deletes    = col("operation") == 'DELETE',
    except_column_list  = ["operation", "sequence_num"],
    stored_as_scd_type  = 2
)

In [0]:
%sql
-- === PIPELINE SOURCE CODE (SQL equivalent) ===
-- Attach this notebook to a Lakeflow Pipeline

-- Bronze: stream new CSV files from landing volume (Auto Loader enabled via STREAM keyword)
CREATE OR REFRESH STREAMING TABLE maintenance_bronze_sql
AS SELECT *
FROM STREAM read_files(
    '/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_*.csv',
    format => 'csv',
    header => true
);

-- Silver: apply CDC turbine status changes as SCD Type 2
CREATE OR REFRESH STREAMING TABLE turbines_silver_sql;

CREATE FLOW turbines_cdc_flow
AS AUTO CDC INTO turbines_silver_sql
FROM STREAM(trainer_demo.demo_07.turbine_events)
KEYS (turbine_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num)
STORED AS SCD TYPE 2;


#### 지역 데이터 통합을 위해 여러 흐름 사용
GreenGrid는 각 지역 그리드 운영자로부터 별도로 터빈 판독값을 수신합니다. **명시적 흐름**은 `UNION` 절 없이 단일 골드 레이어 테이블로 병합됩니다.
```sql
-- Target table for all regions combined
CREATE OR REFRESH STREAMING TABLE turbine_readings_global;

-- Each regional flow appends independently
CREATE FLOW readings_europe_flow
AS INSERT INTO turbine_readings_global BY NAME
SELECT *, 'Europe' AS region
FROM STREAM(bronze_readings_europe);

CREATE FLOW readings_north_america_flow
AS INSERT INTO turbine_readings_global BY NAME
SELECT *, 'North America' AS region
FROM STREAM(bronze_readings_north_america);
```

이 패턴은 `UNION`보다 더 유연합니다. 기존 흐름을 수정하거나 다시 시작하지 않고도 새 흐름을 추가하여 새 지역을 추가할 수 있습니다.